In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4050 Laptop GPU


In [ ]:
"""
Temporal Fusion Transformer for continuous remaining-lap-time prediction
with quantile uncertainty (P10/P50/P90).

*** EXPECTED INPUT SCHEMA (this does NOT exist yet -- feature engineering
    step still needs to build this from your raw fastf1_pull.py output) ***

A single DataFrame with one row per telemetry sample, containing:

  group_id columns (identify one continuous time series = one lap):
    - "session_id"     e.g. "2023_R09"      (str)
    - "driver"         driver number         (str, categorical)
    - "lap_number"     int

  time_idx (REQUIRED by pytorch-forecasting):
    - "time_idx"       integer step within the lap, starting at 0,
                        strictly increasing, no gaps (resample telemetry
                        to a fixed rate, e.g. 10Hz, to guarantee this)

  target:
    - "remaining_lap_time"   seconds remaining until lap completion
                              (float; this is what we predict)

  static categoricals (constant across a whole lap):
    - "team", "circuit", "compound_at_stint_start", "event_format"

  static reals:
    - "tyre_age_at_stint_start"

  known future inputs (knowable in advance -- doesn't require a forecast):
    - "remaining_distance"      total_lap_distance - distance_so_far
    - "distance_into_lap"

  observed past inputs (actual telemetry, only known up to "now"):
    - "speed", "throttle", "brake_pct", "gear", "drs",
      "air_temp", "track_temp", "humidity", "wind_speed", "tyre_life"

Run:
    python train_tft.py
"""

import logging
from pathlib import Path

import pandas as pd
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("train_tft")

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
FEATURE_TABLE_DIR = Path("data/features")  # written by build_features.py, one parquet per session
MODEL_OUTPUT_DIR = Path("models/tft")

MAX_ENCODER_LENGTH = 50    # how much history (timesteps) the model looks back on
MAX_PREDICTION_LENGTH = 1  # we predict a single point (remaining time), not multi-step

QUANTILES = [0.1, 0.5, 0.9]
BATCH_SIZE = 128
MAX_EPOCHS = 50
LEARNING_RATE = 0.03        # pytorch-forecasting's LR finder often lands near here for TFT; tune via find_lr below
GRADIENT_CLIP_VAL = 0.1     # TFT is prone to exploding gradients; keep this tight

GROUP_IDS = ["session_id", "driver", "lap_number"]
STATIC_CATEGORICALS = ["team", "circuit", "compound_at_stint_start", "event_format"]
STATIC_REALS = ["tyre_age_at_stint_start"]
KNOWN_REALS = ["remaining_distance", "distance_into_lap"]
UNKNOWN_REALS = [
    "speed", "throttle", "brake_pct", "gear", "drs",
    "air_temp", "track_temp", "humidity", "wind_speed", "tyre_life",
]
TARGET = "remaining_lap_time"


SEASONS_TO_LOAD = None  # e.g. [2023, 2024] to prototype on a subset first; None = all available
ROUNDS_TO_LOAD = [("2024", "8"), ("2025", "8")]    # e.g. [("2023", "1"), ("2025", "1")] for a minimal smoke test --
                          # needs at least one pre-2025 round (train) and one 2025 round (test)
                          # to actually exercise the full train/val/holdout pipeline

FLOAT_COLS = ["remaining_distance", "distance_into_lap", "speed", "throttle", "brake_pct",
              "gear", "drs", "air_temp", "track_temp", "humidity", "wind_speed", "tyre_life",
              "tyre_age_at_stint_start", TARGET]
CATEGORY_COLS = ["session_id", "driver", "team", "circuit", "compound_at_stint_start", "event_format"]


def load_feature_table() -> pd.DataFrame:
    if not FEATURE_TABLE_DIR.exists() or not any(FEATURE_TABLE_DIR.glob("*.parquet")):
        raise FileNotFoundError(
            f"{FEATURE_TABLE_DIR} has no parquet files yet -- run build_features.py first "
            "(see its module docstring for what it produces)."
        )

    required_cols = list(set(
        GROUP_IDS + STATIC_CATEGORICALS + STATIC_REALS
        + KNOWN_REALS + UNKNOWN_REALS + [TARGET, "time_idx", "season", "lap_number"]
    ))

    files = sorted(FEATURE_TABLE_DIR.glob("*.parquet"))
    if SEASONS_TO_LOAD is not None:
        files = [f for f in files if any(f"year={s}" in f.name for s in SEASONS_TO_LOAD)]
        if not files:
            raise FileNotFoundError(f"No feature files matched SEASONS_TO_LOAD={SEASONS_TO_LOAD}")
    if ROUNDS_TO_LOAD is not None:
        files = [
            f for f in files
            if any(f"year={y}_round={r}.parquet" == f.name for y, r in ROUNDS_TO_LOAD)
        ]
        if not files:
            raise FileNotFoundError(f"No feature files matched ROUNDS_TO_LOAD={ROUNDS_TO_LOAD}")

    if SEASONS_TO_LOAD is None and ROUNDS_TO_LOAD is None and len(files) > 5:
        logger.warning(
            f"!!! About to load ALL {len(files)} feature files with no SEASONS_TO_LOAD or "
            f"ROUNDS_TO_LOAD scoping set. Based on prior runs, this can mean 40+ GB in memory "
            f"and WILL likely crash on most machines. If this is intentional, ignore this "
            f"warning. If not, stop now (Ctrl+C / interrupt the kernel) and set "
            f"ROUNDS_TO_LOAD = [(\"2023\", \"1\"), (\"2025\", \"1\")] near the top of this file."
        )

    logger.info(f"Loading {len(files)} feature files (columns limited to what training needs)...")

    frames = []
    for f in files:
        # only read the columns training actually needs -- avoids paying for
        # any extra columns build_features.py might have carried along
        frame = pd.read_parquet(f, columns=required_cols)

        # downcast immediately, per-file, before concatenating -- concatenating
        # float64 first and downcasting after still spikes peak memory usage
        for col in FLOAT_COLS:
            if col in frame.columns:
                frame[col] = frame[col].astype("float32")
        for col in CATEGORY_COLS:
            if col in frame.columns:
                frame[col] = frame[col].astype("category")

        frames.append(frame)

    df = pd.concat(frames, ignore_index=True)
    logger.info(
        f"Loaded {len(df):,} rows, "
        f"{df.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory"
    )

    required_cols_check = set(GROUP_IDS + STATIC_CATEGORICALS + STATIC_REALS
                               + KNOWN_REALS + UNKNOWN_REALS + [TARGET, "time_idx", "season"])
    missing = required_cols_check - set(df.columns)
    if missing:
        raise ValueError(f"Feature table is missing required columns: {missing}")

    return df


def build_datasets(df: pd.DataFrame):
    # Temporal holdout, per our earlier design: train on 2022-2024, hold out
    # 2025 entirely. This is a season-level split, not a row-level random
    # split -- the whole point is testing generalization to an unseen season.
    train_df = df[df["season"] < 2025].copy()
    test_df = df[df["season"] == 2025].copy()

    if train_df.empty or test_df.empty:
        raise ValueError(
            "Temporal split produced an empty train or test set -- check that "
            "'season' actually spans 2022-2025 in your feature table."
        )

    from pytorch_forecasting.data.encoders import NaNLabelEncoder

    training = TimeSeriesDataSet(
        train_df,
        time_idx="time_idx",
        target=TARGET,
        group_ids=GROUP_IDS,
        max_encoder_length=MAX_ENCODER_LENGTH,
        max_prediction_length=MAX_PREDICTION_LENGTH,
        static_categoricals=STATIC_CATEGORICALS,
        static_reals=STATIC_REALS,
        time_varying_known_reals=KNOWN_REALS,
        time_varying_unknown_reals=UNKNOWN_REALS,
        target_normalizer=GroupNormalizer(groups=GROUP_IDS, transformation="softplus"),
        categorical_encoders={
            "session_id": NaNLabelEncoder(add_nan=True),
            "driver": NaNLabelEncoder(add_nan=True),
            "team": NaNLabelEncoder(add_nan=True),
            "circuit": NaNLabelEncoder(add_nan=True),
        },
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )

    # validation: same structure, built from the training dataset's encoders
    # so categorical mappings stay consistent (don't fit a new one on val/test)
    validation = TimeSeriesDataSet.from_dataset(
        training, train_df, predict=True, stop_randomization=True
    )

    # 2025 holdout uses the SAME encoders/normalizers fit on training data --
    # this is important: fitting fresh encoders on test data would leak
    # information and also silently break if 2025 has categories unseen in
    # training (e.g. a driver who wasn't racing in 2022-2024).
    test_dataset = TimeSeriesDataSet.from_dataset(
        training, test_df, predict=True, stop_randomization=True
    )

    return training, validation, test_dataset


def build_model(training: TimeSeriesDataSet) -> TemporalFusionTransformer:
    return TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=LEARNING_RATE,
        hidden_size=32,              # start modest; this is a common first-pass size for TFT
        attention_head_size=4,
        dropout=0.2,
        hidden_continuous_size=16,
        loss=QuantileLoss(quantiles=QUANTILES),
        log_interval=10,
        reduce_on_plateau_patience=4,
    )


def main():
    logger.info(f"DEBUG: ROUNDS_TO_LOAD={ROUNDS_TO_LOAD!r}, SEASONS_TO_LOAD={SEASONS_TO_LOAD!r}")
    logger.info("Loading feature table...")
    df = load_feature_table()
    logger.info(f"Loaded {len(df):,} rows across {df[GROUP_IDS].drop_duplicates().shape[0]:,} groups, "
                f"{df.memory_usage(deep=True).sum()/1e9:.2f} GB")

    logger.info("Building TimeSeriesDataSets (train / val / 2025 holdout)...")
    training, validation, test_dataset = build_datasets(df)

    train_loader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=2, persistent_workers=True)
    val_loader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=2, persistent_workers=True)

    model = build_model(training)
    logger.info(f"Model parameter count: {sum(p.numel() for p in model.parameters()):,}")

    early_stop = EarlyStopping(monitor="val_loss", patience=8, mode="min")
    checkpoint = ModelCheckpoint(
        dirpath=MODEL_OUTPUT_DIR,
        filename="tft-{epoch:02d}-{val_loss:.4f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator="auto",
        gradient_clip_val=GRADIENT_CLIP_VAL,
        callbacks=[early_stop, checkpoint],
        enable_progress_bar=False,
        log_every_n_steps=1,
    )

    logger.info("Starting training...")
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

    logger.info(f"Best model checkpoint: {checkpoint.best_model_path}")

    # --- Evaluate on the 2025 holdout ---
    logger.info("Evaluating on 2025 holdout season...")
    best_model = TemporalFusionTransformer.load_from_checkpoint(checkpoint.best_model_path)
    test_loader = test_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=0)

    predictions = best_model.predict(test_loader, mode="quantiles", return_x=True)
    logger.info(f"Holdout predictions shape: {predictions.output.shape}")  # [n_samples, pred_len, n_quantiles]

    # NOTE: calibration check goes here as a follow-up step, not in this script --
    # verify what fraction of true values actually fall inside the P10-P90 band
    # on this holdout before trusting the uncertainty estimates.

    logger.info("Done.")


if __name__ == "__main__":
    main()

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "d:\Material\Programming\Machine Learning\F1 LapTime Prediction\f1-laptime-prediction\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
%pip uninstall torch torchvision

In [ ]:
%pip install torch torchvision